<a href="https://colab.research.google.com/github/Karn2898/GenAI_Learning_Archive/blob/main/Llamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LlamaIndex


In [19]:
pip install llama-index-llms-google-genai llama-index-embeddings-google-genai

In [5]:
pip install --upgrade llama-index llama-index-core

In [2]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex, ServiceContext, StorageContext
from llama_index.core import load_index_from_storage, StorageContext
from llama_index.llms.google_genai import GoogleGenerativeAI
from llama_index.embeddings.google_genai import GoogleGeminiEmbedding
import os

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


load data


In [3]:
!mkdir data

mkdir: cannot create directory ‘data’: File exists


In [4]:
documents= SimpleDirectoryReader("data").load_data()

In [5]:
documents

[Document(id_='fd48ed2f-41c3-4308-87be-b6740e33db3e', embedding=None, metadata={'page_label': '1', 'file_name': 'paper-yolo.pdf', 'file_path': '/content/data/paper-yolo.pdf', 'file_type': 'application/pdf', 'file_size': 4368906, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Y ou Only Look Once:\nUniﬁed, Real-Time Object Detection\nJoseph Redmon⇤, Santosh Divvala⇤†, Ross Girshick¶, Ali Farhadi⇤†\nUniversity of Washington⇤, Allen Institute for AI †, Facebook AI Research ¶\nhttp://pjreddie.com/yolo/\nAbstract\nWe present YOLO, a new approach to object detection.\nPrior work

split texts into chunks

In [6]:
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

In [11]:
llm=GoogleGenerativeAI(model="gemini-pro")

/tmp/ipykernel_8066/3137919986.py:1: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  llm=Gemini()


In [25]:
import google.generativeai as genai

genai.configure(api_key="GOOGLE_API_KEY")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [26]:
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [38]:
from llama_index.core import Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding

Settings.llm = Gemini(model="models/gemini-2.5-flash")

Settings.embed_model = GeminiEmbedding(model_name="models/gemini-embedding-001")
Settings.chunk_size = 800
Settings.chunk_overlap = 20

/tmp/ipykernel_8066/3532969623.py:5: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(model="models/gemini-2.5-flash")
/tmp/ipykernel_8066/3532969623.py:7: DeprecationWarning: Call to deprecated class GeminiEmbedding. (Should use `llama-index-embeddings-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/embeddings/google_genai/Support for this package will be discontinued as of v0.4.2) -- Deprecated since version 0.4.2.
  Settings.embed_model = GeminiEmbedding(model_name="models/gemini-embedding-001")


In [39]:
Index=VectorStoreIndex.from_documents(documents)

Load Index

In [40]:
Index.storage_context.persist()

In [41]:
storage_context=StorageContext.from_defaults(persist_dir='./storage')
Index=load_index_from_storage(storage_context=storage_context)

Q/A

In [43]:
query_engine = Index.as_query_engine()

In [48]:
response=query_engine.query("what is yolo?")
response

Response(response='YOLO is a novel approach to object detection that reframes the task as a single regression problem. It utilizes a single convolutional neural network to directly predict bounding boxes and their associated class probabilities from entire images in one evaluation. This unified architecture is optimized end-to-end for detection performance.', source_nodes=[NodeWithScore(node=TextNode(id_='6932ea9c-9059-4d61-acb1-15b0d4df90b5', embedding=None, metadata={'page_label': '1', 'file_name': 'paper-yolo.pdf', 'file_path': '/content/data/paper-yolo.pdf', 'file_type': 'application/pdf', 'file_size': 4368906, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: Re

A 429 error means too many requests in a short time.

In [45]:
from IPython.display import Markdown , display

In [49]:
display(Markdown(f"<b>{response}</b"))

<b>YOLO is a novel approach to object detection that reframes the task as a single regression problem. It utilizes a single convolutional neural network to directly predict bounding boxes and their associated class probabilities from entire images in one evaluation. This unified architecture is optimized end-to-end for detection performance.</b